# Generation and evaluation (debug)

Goal:
- reuse the retrieval pipeline built earlier
- retrieve top-k chunks for each question
- build a prompt from the retrieved context
- generate an answer with Llama 3.2 1B
- evaluate predictions with F1, EM, Recall@k, and latency

Important:
- this notebook is for debugging and validating the full pipeline on a small subset first (Testing on CPU directly from the notebook)
- the final large scale experiment will later be moved to a Python script and run with Slurm on GPU (To have access to GPUs with LMU servers are only via slurm)
- retrieval settings to evaluate later:
  - chunk_size in {32, 128, 256}
  - top_k in {1, 5, 10}

In [1]:
# Imports for data loading, indexing, generation, and evaluation

import json
import time
import re
import string
import numpy as np
import pandas as pd
import faiss

from pathlib import Path
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

/home/a/arfaoui/miniconda3/envs/AAML/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

data_dir = Path("/home/a/arfaoui/rag_project/data")

# Retrieval settings that will later be varied systematically
chunk_sizes = [32, 128, 256]
top_k_values = [1, 5, 10]

# For testing, we start with one configuration
debug_chunk_size = 128
debug_top_k = 5

# Start with a small number of questions for fast testing
debug_n_questions = 5
# print to check
print("Data directory:", data_dir)
print("Test chunk size:", debug_chunk_size)
print("Test top-k:", debug_top_k)
print("Test number of questions:", debug_n_questions)

Data directory: /home/a/arfaoui/rag_project/data
Test chunk size: 128
Test top-k: 5
Test number of questions: 5


In [3]:
# Load saved embeddings and metadata for all chunk sizes
# We need these to rebuild or reuse the retrieval pipeline inside this notebook 

all_embeddings = {}
all_metadata = {}

for size in chunk_sizes:
    # Load embedding matrix
    emb_path = data_dir / f"embeddings_{size}.npy"
    embeddings = np.load(emb_path)
    all_embeddings[size] = embeddings

    # Load chunk metadata
    meta_path = data_dir / f"chunks_metadata_{size}.json"
    with open(meta_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    all_metadata[size] = metadata

    print(f"\nchunk_size={size}")
    print("Embeddings shape:", embeddings.shape)
    print("Metadata entries:", len(metadata))


chunk_size=32
Embeddings shape: (22186, 384)
Metadata entries: 22186

chunk_size=128
Embeddings shape: (7295, 384)
Metadata entries: 7295

chunk_size=256
Embeddings shape: (5236, 384)
Metadata entries: 5236


In [4]:
# Load precomputed FAISS indexes from disk

faiss_indexes = {}

for size in chunk_sizes:
    index_path = data_dir / f"faiss_index_{size}.index"
    index = faiss.read_index(str(index_path))
    
    faiss_indexes[size] = index
    print(f"Loaded FAISS index for chunk_size={size}")
    print("Index size:", index.ntotal)

Loaded FAISS index for chunk_size=32
Index size: 22186
Loaded FAISS index for chunk_size=128
Index size: 7295
Loaded FAISS index for chunk_size=256
Index size: 5236


In [5]:
# Load the fixed HotpotQA sample from disk
# The sample file is in JSON Lines format, so we use the datasets library.

sample_path = data_dir / "hotpotqa_sample_500.json"
hotpot_sample = load_dataset("json", data_files=str(sample_path))["train"]
# print to check 
print("Loaded questions:", len(hotpot_sample))
print("Example question:", hotpot_sample[0]["question"])
print("Example answer:", hotpot_sample[0]["answer"])
# Create a small testing subset of questions
# We use only the first N questions for quick pipeline testing.

debug_sample = hotpot_sample.select(range(debug_n_questions))

print("Debug subset size:", len(debug_sample))
print("First debug question:", debug_sample[0]["question"])

Loaded questions: 500
Example question: What date in 2010 was a South Korean film starring  Kim Hyang-gi released?
Example answer: January 14, 2010
Debug subset size: 5
First debug question: What date in 2010 was a South Korean film starring  Kim Hyang-gi released?


In [6]:
# Load the same embedding model used earlier for chunk embeddings
# We need it here to embed questions for retrieval.

embedding_model_name = "BAAI/bge-small-en-v1.5"
embed_model = SentenceTransformer(embedding_model_name)

print("Embedding model loaded:", embedding_model_name)

Embedding model loaded: BAAI/bge-small-en-v1.5


In [7]:
# Same function from notebook 05 to retrieve top-k chunks for a given question

def retrieve_top_k(question, index, metadata, k=5):
    """
    Given a question, retrieve top-k most similar chunks.
    
    Returns:
    - retrieved_chunks: list of metadata entries
    - scores: similarity scores
    """
    
    # BGE models expect "query: " prefix for queries
    query_text = "query: " + question
    
    # Encode query
    query_embedding = embed_model.encode([query_text])
    
    # Convert to float32 (FAISS requirement)
    query_embedding = np.array(query_embedding).astype("float32")
    
    # Search in FAISS index
    D, I = index.search(query_embedding, k)
    
    # Retrieve corresponding chunks
    retrieved_chunks = [metadata[idx] for idx in I[0]]
    scores = D[0]
    
    return retrieved_chunks, scores

In [8]:
# Build a single text context string from retrieved chunks
# This will later be passed into the LLM prompt, without it the retrieved chunks are useful information but can't give it to the model
# This makes it:
#  -human-readable
#  -structured for the LLM
#  -easy to debug


def build_retrieved_context(retrieved_chunks):
    """
    Concatenate retrieved chunks into one context string.
    """
    parts = []

    for i, chunk in enumerate(retrieved_chunks, start=1):
        parts.append(f"[Chunk {i}] Title: {chunk['title']}\n{chunk['text']}")

    return "\n\n".join(parts)

## LLM generation setup

In this section, we load the fixed generation model and define a prompt template.

Important:
- the LLM is fixed across all experiments
- the prompt must remain constant across all configurations
- the model should answer using only the retrieved context
- this stage is still for testing on a small subset before the final Slurm run  
  
## Prompt design

We use one fixed prompt template across all experiments.

Design goals:
- force the model to rely only on retrieved context
- keep the prompt simple and constant

In [9]:
# Load the generation model and tokenizer
# We keep the model fixed across all experiments so that we isolate the effect of the chunk size and top-k 

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

llm_name = "meta-llama/Llama-3.2-1B-Instruct"

# Load tokenizer
llm_tokenizer = AutoTokenizer.from_pretrained(llm_name)

# Load causal language model
# device_map="auto" lets transformers place the model on available hardware
# torch_dtype="auto" uses a suitable dtype automatically when possible
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype="auto",
    device_map="auto"
)
# print to check 
print("LLM tokenizer loaded:", llm_name)
print("LLM model loaded:", llm_name)

/home/a/arfaoui/miniconda3/envs/AAML/lib/python3.10/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


LLM tokenizer loaded: meta-llama/Llama-3.2-1B-Instruct
LLM model loaded: meta-llama/Llama-3.2-1B-Instruct


In [10]:
# Build a fixed prompt from retrieved context and question
# We want the model to extract a short answer from the context.

def build_prompt(question, context):
    prompt = f"""Context:
{context}

Question: {question}

Short answer (just the key phrase, no full sentences):"""
    return prompt
    

In [11]:
# Test retrieval + context building + prompt construction on one example question

example = debug_sample[0]

question = example["question"]
correct_answer = example["answer"]

# Use the balanced debug setting first
index = faiss_indexes[debug_chunk_size]
metadata = all_metadata[debug_chunk_size]

retrieved_chunks, retrieval_scores = retrieve_top_k(
    question=question,
    index=index,
    metadata=metadata,
    k=debug_top_k
)

context = build_retrieved_context(retrieved_chunks)
prompt = build_prompt(question, context)

print("Question:", question)
print("correct ansewer:", correct_answer)
print("\n--- Prompt preview ---\n")
print(prompt[:2000])  # preview first part only

Question: What date in 2010 was a South Korean film starring  Kim Hyang-gi released?
correct ansewer: January 14, 2010

--- Prompt preview ---

Context:
[Chunk 1] Title: Kim Hyang-gi
Kim Hyang-gi (born August 9, 2000) is a South Korean actress.  Kim began her career as a child actress, and has starred in films and television series such as "Wedding Dress" (2010), "The Queen's Classroom" (2013), "Thread of Lies" (2014) and "Snowy Road" (2017).

[Chunk 2] Title: Wedding Dress (film)
Wedding Dress is a South Korean drama film, released on January 14, 2010.

[Chunk 3] Title: Blood Rain (film)
Blood Rain () is a 2005 South Korean film.  A murder mystery set in 1808, it touches on historical prejudice against Roman Catholicism in the Joseon Kingdom.  Although primarily a period thriller, director Kim Dae-seung weaves together an unconventional mix of styles—a puzzle-box mystery plot traditionally associated with detective fiction, class-conscious social commentary, lush cinematography, sets 

In [12]:
# Generate an answer from the LLM given a prompt
# Decode only the newly generated continuation, not the entire prompt.

def generate_answer(prompt, tokenizer, model, max_new_tokens=32):
    """
    Generate a short answer from the LLM.
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Only decode the newly generated tokens
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # Keep only the first line
    answer = answer.split("\n")[0].strip()

    return answer

In [13]:
# Run one full pipeline example: retrieval → prompt → generation

start_time = time.time()

generated_answer = generate_answer(
    prompt,
    tokenizer=llm_tokenizer,
    model=llm_model
)

latency = time.time() - start_time

print("=== MODEL OUTPUT ===\n")
print(generated_answer)

print("\n=== LATENCY ===")
print(f"{latency:.4f} seconds")

/home/a/arfaoui/miniconda3/envs/AAML/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/a/arfaoui/miniconda3/envs/AAML/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


=== MODEL OUTPUT ===

January 14, 2010.

=== LATENCY ===
24.6208 seconds


# Observation
The prediction of the LLM is  the right answer as we can see but the formulation can lead to wrong metrics and effect the EM HOW?:
  - the predicted answer contain for example "." but the truth from data sample doesn't contain "."  which can break the exact EM  
**How to slove this**  
Normalize the text:
  - Lowercase
  - remove punctuation
  - ....   
**With text normalization predection and answer should match** 

In [14]:
import string

def normalize_text(text):
    """
    Normalize text for fair comparison.

    - lowercase
    - remove punctuation
    - strip whitespace
    """

    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.strip()

    return text

In [15]:
norm_pred = normalize_text(generated_answer )
norm_gt = normalize_text(correct_answer)

print("Normalized prediction:", norm_pred)
print("Normalized correct_answer:", norm_gt)

Normalized prediction: january 14 2010
Normalized correct_answer: january 14 2010


In [16]:
# Evaluation functions

# Exact Match after normalization
# Returns 1 if prediction and correct answer match exactly, else 0.

def compute_em(prediction, correct_answer):
    pred_norm = normalize_text(prediction)
    gt_norm = normalize_text(correct_answer)
    return int(pred_norm == gt_norm)
    
# Token-level F1 score
# This measures overlap between predicted answer tokens and correct answer tokens.

def compute_f1(prediction, correct_answer):
    pred_tokens = normalize_text(prediction).split()
    gt_tokens = normalize_text(correct_answer).split()

    # Handle empty cases safely
    if len(pred_tokens) == 0 and len(gt_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return 0.0

    # Count token overlap
    common_tokens = set(pred_tokens) & set(gt_tokens)
    num_same = sum(min(pred_tokens.count(tok), gt_tokens.count(tok)) for tok in common_tokens)

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)
    f1 = 2 * precision * recall / (precision + recall)

    return f1
    
# Recall@k for retrieval
# Returns 1 if the answer appears in at least one retrieved chunk, else 0.

def compute_recall_at_k(retrieved_chunks, correct_answer):
    gt_norm = normalize_text(correct_answer)

    for chunk in retrieved_chunks:
        chunk_text_norm = normalize_text(chunk["text"])
        if gt_norm in chunk_text_norm:
            return 1

    return 0
# Recall@k_titles 
# Unlike the simple answer-based Recall@k (which checks if the answer string
# appears in retrieved chunks), this metric evaluates whether the retriever
# successfully retrieved the correct supporting documents.
#
# In HotpotQA, each question is associated with one or more supporting titles
# (documents) required to answer the question. We consider retrieval successful
# if at least one of the retrieved chunks comes from one of these gold titles.
    
def compute_recall_at_k_supporting_titles(retrieved_chunks, supporting_facts):
    """
    Recall@k based on supporting document titles (HotpotQA).

    Returns:
    1 if at least one retrieved chunk has a title matching a gold supporting title
    0 otherwise
    """

    # Extract gold titles and remove duplicates using a set
    gold_titles = set(supporting_facts["title"])

    # Check if any retrieved chunk matches one of the gold titles
    for chunk in retrieved_chunks:
        retrieved_title = chunk.get("title", None)

        if retrieved_title in gold_titles:
            return 1

    return 0

In [17]:
# Test metrics on the current single example

em = compute_em(generated_answer, correct_answer)
f1 = compute_f1(generated_answer, correct_answer)
recall_k = compute_recall_at_k(retrieved_chunks, correct_answer)
recall_title = compute_recall_at_k_supporting_titles( retrieved_chunks, example["supporting_facts"])

print("Generated answer:", generated_answer)
print("Correct answer:", correct_answer)
print("EM:", em)
print("F1:", f1)
print("Recall@k:", recall_k)
print("Recall@k (support titles):", recall_title)

Generated answer: January 14, 2010.
Correct answer: January 14, 2010
EM: 1
F1: 1.0
Recall@k: 1
Recall@k (support titles): 1


In [18]:
def run_single_example(example, chunk_size, top_k):
    """
    Run full RAG pipeline for one example.

    Returns a dictionary with:
    - prediction
    - correct answer
    - EM
    - F1
    - Recall@k
    - latency
    """

    question = example["question"]
    correct_answer = example["answer"]

    # Select correct index + metadata for this chunk size
    index = faiss_indexes[chunk_size]
    metadata = all_metadata[chunk_size]

    # --- Retrieval ---
    retrieved_chunks, _ = retrieve_top_k(
        question=question,
        index=index,
        metadata=metadata,
        k=top_k
    )

    # --- Context building ---
    context = build_retrieved_context(retrieved_chunks)

    # --- Prompt ---
    prompt = build_prompt(question, context)

    # --- Generation + latency ---
    start_time = time.time()

    prediction = generate_answer(
        prompt,
        tokenizer=llm_tokenizer,
        model=llm_model
    )

    latency = time.time() - start_time

    # --- Metrics ---
    em = compute_em(prediction, correct_answer)
    f1 = compute_f1(prediction, correct_answer)
    recall_k = compute_recall_at_k(retrieved_chunks, correct_answer)
    recall_title = compute_recall_at_k_supporting_titles( retrieved_chunks, example["supporting_facts"])

    return {
        "question": question,
        "prediction": prediction,
        "correct_answer": correct_answer,
        "EM": em,
        "F1": f1,
        "Recall@k": recall_k,
        "Recall@k_titles": recall_title,
        "latency": latency
    }

In [19]:
def run_test_evaluation(debug_sample, chunk_size, top_k):
    """
    Run evaluation on a small subset of questions.
    """

    results = []

    for i, example in enumerate(debug_sample):
        print(f"Running example {i+1}/{len(debug_sample)}")

        result = run_single_example(
            example=example,
            chunk_size=chunk_size,
            top_k=top_k
        )

        results.append(result)

    return results

In [20]:
results = run_test_evaluation(
    debug_sample=debug_sample,
    chunk_size=debug_chunk_size,
    top_k=debug_top_k
)


Running example 1/5
Running example 2/5
Running example 3/5
Running example 4/5
Running example 5/5


In [21]:
import numpy as np

def summarize_results(results):
    em = np.mean([r["EM"] for r in results])
    f1 = np.mean([r["F1"] for r in results])
    recall = np.mean([r["Recall@k"] for r in results])
    recall_title = np.mean([r["Recall@k_titles"] for r in results])

    latencies = [r["latency"] for r in results]
    p50 = np.percentile(latencies, 50)
    p95 = np.percentile(latencies, 95)

    return {
        "EM": em,
        "F1": f1,
        "Recall@k": recall,
        "Recall@k_titles": recall_title,
        "Latency_p50": p50,
        "Latency_p95": p95
    }
summary = summarize_results(results)

print("\n=== SUMMARY ===")
for k, v in summary.items():
    print(f"{k}: {v:.4f}")


=== SUMMARY ===
EM: 0.2000
F1: 0.2000
Recall@k: 0.8000
Recall@k_titles: 1.0000
Latency_p50: 12.9557
Latency_p95: 17.2960


In [22]:
# Inspect each example in detail

for i, r in enumerate(results):
    print(f"\n===== Example {i+1} =====")
    print("Question:", r["question"])
    print("Prediction:", r["prediction"])
    print("Ground truth:", r["correct_answer"])
    print("EM:", r["EM"])
    print("F1:", r["F1"])
    print("Recall@k:", r["Recall@k"])
    print("Latency:", f"{r['latency']:.2f}s")


===== Example 1 =====
Question: What date in 2010 was a South Korean film starring  Kim Hyang-gi released?
Prediction: January 14, 2010.
Ground truth: January 14, 2010
EM: 1
F1: 1.0
Recall@k: 1
Latency: 10.35s

===== Example 2 =====
Question: Who was the film which was Kim Dae-woo's directing debut about ?
Prediction: Obsessed
Ground truth: a scholar
EM: 0
F1: 0.0
Recall@k: 1
Latency: 12.96s

===== Example 3 =====
Question: What is the full name of the company co founded by Jay Van Andel?
Prediction: Amway Corporation
Ground truth: American Way
EM: 0
F1: 0.0
Recall@k: 1
Latency: 15.33s

===== Example 4 =====
Question: Who is the author of the play that was adapted into a film and featured the orchestral arrangement Suite from Henry V?
Prediction: Henry James.
Ground truth: William Shakespeare
EM: 0
F1: 0.0
Recall@k: 1
Latency: 17.79s

===== Example 5 =====
Question: what 6-story building was listed on the National Register of Historic Places in 1980?
Prediction: James Bruce Round Barn

In [23]:
# Inspect failing cases with retrieved context

for i, example in enumerate(debug_sample):
    result = results[i]

    if result["EM"] == 0:
        print(f"\n===== FAILING EXAMPLE {i+1} =====")
        print("Question:", result["question"])
        print("Prediction:", result["prediction"])
        print("Ground truth:", result["correct_answer"])

        # Re-run retrieval so we can inspect the chunks
        retrieved_chunks, scores = retrieve_top_k(
            question=result["question"],
            index=faiss_indexes[debug_chunk_size],
            metadata=all_metadata[debug_chunk_size],
            k=debug_top_k
        )

        for j, (chunk, score) in enumerate(zip(retrieved_chunks, scores)):
            print(f"\n[Chunk {j+1}] Score: {score:.4f}")
            print("Title:", chunk["title"])
            print("Text preview:", chunk["text"][:500])


===== FAILING EXAMPLE 2 =====
Question: Who was the film which was Kim Dae-woo's directing debut about ?
Prediction: Obsessed
Ground truth: a scholar

[Chunk 1] Score: 0.8131
Title: Kim Dae-woo
Text preview: Kim Dae-woo (born 1962) is a South Korean film director and screenwriter.  Kim started his filmmaking career by winning the 1991 Korean Film Council Screenplay Contest.  He was an accomplished screenwriter with a number of hit scripts, including "The Girl for Love and The One for Marriage" (1993), "An Affair" (1998), "Rainbow Trout" (1999), and "Untold Scandal" (2003).  Making a switch to directing, he debuted with the hit period drama film "Forbidden Quest" (2006), followed by "The Serv

[Chunk 2] Score: 0.7195
Title: Obsessed (2014 film)
Text preview: Obsessed (; lit.  "Human Addiction" or "Human Intoxication") is a 2014 South Korean erotic romance film written and directed by Kim Dae-woo, about a couple having a passionate affair in a military camp under tight surveillance in 1

# Observation:
During testing and experimentation, I realized that the evaluation metrics could be enhanced by incorporating additional frameworks commonly used for assessing LLMs and RAG systems, such as BERTScore and RAGAS. For this reason, I decided to store the outputs of the main experiment runs—including the question, ground truth, prediction, metrics, and configuration settings. This allows me to later evaluate different configurations and add supplementary metrics offline without needing to rerun the generation process.